In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/cortexmnist")

print("Path to dataset files:", path)

In [1]:
import torch
from ConvLSTM.convlstm import ConvLSTM
import torch.nn as nn

In [2]:
data_train= torch.load('cortex_mnist/train.pt')
data_val= torch.load('cortex_mnist/val.pt')
data_test= torch.load('cortex_mnist/test.pt')


In [3]:
print(data_test.keys())

dict_keys(['spikes_on', 'spikes_off', 'labels'])


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

#class SpikesDataset(Dataset):
#    def __init__(self, data_dict):
#        self.spikes_on  = torch.tensor(data_dict['spikes_on'],  dtype=torch.int32)
#        self.spikes_off = torch.tensor(data_dict['spikes_off'], dtype=torch.float32)
#        self.labels     = torch.tensor(data_dict['labels'],     dtype=torch.long)
#        self.spikes_on.unsqueeze_(1)
#        self.spikes_off.unsqueeze_(1)
#        self.labels.unsqueeze_(1)
#    def __len__(self):
#        return len(self.labels)
#    def __getitem__(self, idx):
#        return {
#            'spikes_on':  self.spikes_on[idx],
#            'spikes_off': self.spikes_off[idx],
#            'label':      self.labels[idx]
#        }

class SpikesDataset(Dataset):
    def __init__(self, data_dict):
        # Store as numpy arrays — no tensor conversion yet
        self.spikes_on = data_dict['spikes_on']   # raw numpy
        self.labels    = data_dict['labels']       # raw numpy

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        spikes_on = torch.tensor(self.spikes_on[idx], dtype=torch.float32).unsqueeze(0)  # (1, H, W)
        label     = torch.tensor(self.labels[idx],    dtype=torch.float32).unsqueeze(0)  # (1, H, W)
        return {
            'spikes_on': spikes_on,
            'label':     label
        }
dataset_train=SpikesDataset(data_train)
dataset_val=SpikesDataset(data_val)
dataset_test=SpikesDataset(data_test)

train_loader=DataLoader(dataset=dataset_train,batch_size=32,shuffle=True,num_workers=4)
val_loader=DataLoader(dataset=dataset_val,batch_size=32,shuffle=False,num_workers=4)
test_loader=DataLoader(dataset=dataset_test,batch_size=32,shuffle=False,num_workers=4)



In [5]:
class NeuralDecoder(nn.Module):
    def __init__(self):
        super().__init__()                  # ← was super() — must call __init__
        self.decoder = ConvLSTM(
            img_size=(40, 40),
            input_dim=1,
            hidden_dim=16,
            kernel_size=(3, 3),
            cnn_dropout=0.5,
            rnn_dropout=0.5,
            batch_first=True,
            bias=True,
            peephole=False,
            layer_norm=False,
            return_sequence=True,
            bidirectional=True
        )

    def forward(self, x):
        _, last_state, _ = self.decoder(x)  # last_state: (B, 2, H, W) bidirectional
        return last_state.mean(dim=1, keepdim=True)  # (B, 1, H, W) ← matches label shape

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Training + Validation Pipeline ────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        spikes_on = batch['spikes_on'].to(device)  # (B, 1, H, W)
        labels    = batch['label'].to(device)      # (B, 1, H, W)

        optimizer.zero_grad()
        out  = model(spikes_on)                    # (B, 1, H, W)
        loss = criterion(out.squeeze(1), labels.squeeze(1).float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        spikes_on = batch['spikes_on'].to(device)
        labels    = batch['label'].to(device)

        out  = model(spikes_on)
        loss = criterion(out.squeeze(1), labels.squeeze(1).float())
        total_loss += loss.item()
    return total_loss / len(loader)


def train(model, train_loader, val_loader, optimizer, criterion,
          device, num_epochs=50, save_path="best_model.pt"):
    """
    Full training loop with validation.
    Saves the checkpoint with the lowest validation loss.
    Returns a dict with train/val loss histories.
    """
    model.to(device)
    best_val_loss = float('inf')
    history = {'train': [], 'val': []}

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss   = evaluate(model, val_loader, criterion, device)

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        # ── Save best checkpoint ──────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss':             best_val_loss,
            }, save_path)
            tag = "  ✓ saved"
        else:
            tag = ""

        print(f"Epoch [{epoch:>3}/{num_epochs}]  "
              f"train_loss: {train_loss:.6f}  "
              f"val_loss: {val_loss:.6f}{tag}")

    print(f"\nBest val loss: {best_val_loss:.6f}  →  checkpoint: {save_path}")
    return history


# ── Plotting: ground-truth label vs reconstruction ────────────────────────────

@torch.no_grad()
def plot_reconstructions(model, test_loader, device,
                         num_samples=5, checkpoint_path=None):
    """
    Loads the best checkpoint (optional), runs inference on the test set,
    and plots the first `num_samples` pairs of (label, reconstruction).
    """
    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
              f"(val_loss={ckpt['val_loss']:.6f})")

    model.to(device)
    model.eval()

    # ── Collect samples ───────────────────────────────────────────────────────
    labels_all, recons_all = [], []
    for batch in test_loader:
        spikes_on = batch['spikes_on'].to(device)
        labels    = batch['label']               # keep on CPU for plotting

        out = model(spikes_on).cpu()             # (B, 1, H, W)

        labels_all.append(labels.squeeze(1))     # (B, H, W)
        recons_all.append(out.squeeze(1))        # (B, H, W)

        if sum(x.shape[0] for x in labels_all) >= num_samples:
            break

    labels_all = torch.cat(labels_all)[:num_samples]   # (N, H, W)
    recons_all = torch.cat(recons_all)[:num_samples]   # (N, H, W)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(num_samples, 2,
                             figsize=(5, 2.5 * num_samples),
                             squeeze=False)

    for i in range(num_samples):
        label = labels_all[i].float().numpy()
        recon = recons_all[i].numpy()

        vmin = min(label.min(), recon.min())
        vmax = max(label.max(), recon.max())

        axes[i][0].imshow(label, cmap='gray', vmin=vmin, vmax=vmax)
        axes[i][0].set_title(f"Sample {i+1} — Label", fontsize=9)
        axes[i][0].axis('off')

        axes[i][1].imshow(recon, cmap='gray', vmin=vmin, vmax=vmax)
        axes[i][1].set_title(f"Sample {i+1} — Reconstruction", fontsize=9)
        axes[i][1].axis('off')

    plt.suptitle("Label vs Reconstruction (test set)", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()
    return fig




In [7]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model     = NeuralDecoder()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = train(
    model, train_loader, val_loader,
    optimizer, criterion,
    device     = device,
    num_epochs = 50,
    save_path  = "best_model.pt"
)



(3, 3)


KeyboardInterrupt: 

In [ ]:
fig = plot_reconstructions(
    model, test_loader,
    device           = device,
    num_samples      = 5,
    checkpoint_path  = "best_model.pt"
)